[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/14_busqueda_informada/notebooks/aplicaciones/04_puzzle_8.ipynb)

# Notebook 04: El Puzzle de 8 Piezas

El **puzzle de 8 piezas** (o 8-puzzle) es un rompecabezas clásico de inteligencia artificial. Consiste en un tablero de $3 \times 3$ con 8 fichas numeradas del 1 al 8 y un espacio vacío (representado por 0). El objetivo es llevar el tablero desde una configuración inicial hasta el **estado meta**:

```
1 2 3
4 5 6
7 8 _
```

Las fichas adyacentes al espacio vacío pueden deslizarse hacia él. El espacio de estados tiene $9! / 2 = 181{,}440$ configuraciones alcanzables.

En este notebook compararemos cuatro estrategias:
1. **BFS**: búsqueda en anchura (no informada, costosa)
2. **A\* + fichas fuera de lugar**: heurística simple
3. **A\* + distancia Manhattan**: heurística más informativa
4. **IDA\***: A\* iterativo con ahorro de memoria (ejercicio)

In [ ]:
# Instalar dependencias si es necesario (Colab)
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import heapq
import random
import math
from collections import deque

print("Dependencias listas.")

In [ ]:
# ── Estilo y colores ──────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")

COLORS = {
    "primary":    "#2563EB",
    "secondary":  "#10B981",
    "accent":     "#F59E0B",
    "danger":     "#EF4444",
    "node":       "#93C5FD",
    "edge":       "#475569",
}

plt.rcParams.update({
    "figure.dpi":       120,
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "font.size":        10,
    "legend.fontsize":  9,
})

np.random.seed(42)
random.seed(42)
print("Estilo configurado.")

---
## Sección 1: Representación del Estado

Representaremos el tablero como una **tupla de 9 enteros** (índices 0–8), donde el valor `0` indica el espacio vacío.

La posición $(i, j)$ en el tablero corresponde al índice $3i + j$ en la tupla.

```
índice:  0 1 2
         3 4 5
         6 7 8
```

In [ ]:
# ── Clase PuzzleState ─────────────────────────────────────────────────────────

META = (1, 2, 3, 4, 5, 6, 7, 8, 0)

# Movimientos posibles: (delta_fila, delta_columna) -> nombre
MOVIMIENTOS = [
    (-1, 0, "Arriba"),
    ( 1, 0, "Abajo"),
    ( 0,-1, "Izquierda"),
    ( 0, 1, "Derecha"),
]


class PuzzleState:
    """Estado del puzzle de 8 piezas representado como tupla inmutable."""

    def __init__(self, state):
        """
        state: tuple de 9 enteros (0 = espacio vacío)
        """
        self.state = tuple(state)

    def find_blank(self):
        """Retorna el índice (i, j) del espacio vacío."""
        idx = self.state.index(0)
        return divmod(idx, 3)  # (fila, columna)

    def neighbors(self):
        """
        Genera los estados vecinos moviéndose en las 4 direcciones.
        Retorna lista de (PuzzleState, costo=1, accion).
        """
        vecinos = []
        fi, ci = self.find_blank()
        idx_blank = fi * 3 + ci

        for df, dc, nombre in MOVIMIENTOS:
            nf, nc = fi + df, ci + dc
            if 0 <= nf < 3 and 0 <= nc < 3:
                idx_ficha = nf * 3 + nc
                nuevo_state = list(self.state)
                # Intercambiar el espacio con la ficha
                nuevo_state[idx_blank], nuevo_state[idx_ficha] = (
                    nuevo_state[idx_ficha], nuevo_state[idx_blank]
                )
                vecinos.append((PuzzleState(nuevo_state), 1, nombre))

        return vecinos

    def is_goal(self, meta=META):
        """Verifica si el estado es la meta."""
        return self.state == meta

    def __hash__(self):
        return hash(self.state)

    def __eq__(self, other):
        return self.state == other.state

    def __repr__(self):
        return f"PuzzleState({self.state})"


# ── Función de conteo de inversiones ─────────────────────────────────────────

def contar_inversiones(state):
    """
    Cuenta el número de inversiones en la secuencia (sin contar el 0).
    Una inversión es un par (i, j) con i < j y state[i] > state[j] > 0.
    """
    fichas = [x for x in state if x != 0]
    inversiones = 0
    for i in range(len(fichas)):
        for j in range(i + 1, len(fichas)):
            if fichas[i] > fichas[j]:
                inversiones += 1
    return inversiones


def es_resoluble(state):
    """
    Para tablero 3x3: un estado es resoluble si el número de inversiones es par.
    """
    return contar_inversiones(state) % 2 == 0


def generate_solvable_puzzle(profundidad=10, semilla=None):
    """
    Genera un estado inicial resoluble aplicando `profundidad` movimientos
    aleatorios desde el estado meta. Garantiza que sea resoluble y
    que la solución tenga al menos `profundidad` pasos.
    """
    rng = random.Random(semilla)
    estado = PuzzleState(META)
    prev_estado = None

    for _ in range(profundidad * 3):  # más pasos para mayor aleatoriedad
        vecinos = estado.neighbors()
        # Evitar deshacer el movimiento anterior
        opciones = [v for v in vecinos if v[0] != prev_estado]
        if not opciones:
            opciones = vecinos
        prev_estado = estado
        estado, _, _ = rng.choice(opciones)

    return estado


print("PuzzleState definido.")
print(f"Estado META: {META}")

# Prueba básica
estado_prueba = PuzzleState((1, 2, 3, 4, 0, 5, 7, 8, 6))
print(f"\nEstado de prueba: {estado_prueba.state}")
print(f"Espacio vacío en: {estado_prueba.find_blank()}")
print(f"¿Es meta? {estado_prueba.is_goal()}")
print(f"¿Es resoluble? {es_resoluble(estado_prueba.state)}")
print(f"\nVecinos ({len(estado_prueba.neighbors())} posibles):")
for v, c, a in estado_prueba.neighbors():
    print(f"  Movimiento '{a}': {v.state}")

In [ ]:
# ── Función para visualizar el puzzle ────────────────────────────────────────

def draw_puzzle(state, titulo="", ax=None, meta=META):
    """
    Visualiza el estado del puzzle como una cuadrícula 3x3.
    Las fichas en posición correcta se muestran en verde;
    las incorrectas en azul; el espacio vacío en gris.
    """
    if isinstance(state, PuzzleState):
        state = state.state

    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 3))
    else:
        fig = ax.figure

    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3)
    ax.set_aspect("equal")
    ax.axis("off")

    for idx, valor in enumerate(state):
        fila = 2 - (idx // 3)  # invertimos fila para que 0,0 esté arriba
        col  = idx % 3

        if valor == 0:
            color = "#E5E7EB"  # gris claro para el espacio vacío
            texto = ""
        elif valor == meta[idx]:
            color = COLORS["secondary"]  # verde: ficha en posición correcta
            texto = str(valor)
        else:
            color = COLORS["node"]  # azul: ficha fuera de lugar
            texto = str(valor)

        rect = plt.Rectangle((col, fila), 1, 1,
                               facecolor=color, edgecolor="#1E293B", linewidth=2)
        ax.add_patch(rect)

        if texto:
            ax.text(col + 0.5, fila + 0.5, texto,
                    ha="center", va="center",
                    fontsize=18, fontweight="bold", color="#1E293B")

    if titulo:
        ax.set_title(titulo, fontsize=9, pad=4)

    return fig, ax


# Visualizar estado meta y estado de prueba
fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))

draw_puzzle(META,                        titulo="Estado Meta",    ax=axes[0])
draw_puzzle(estado_prueba,               titulo="Estado Prueba",  ax=axes[1])
draw_puzzle(generate_solvable_puzzle(8), titulo="Aleatorio (d≈8)", ax=axes[2])

leyenda = [
    mpatches.Patch(color=COLORS["secondary"], label="Posición correcta"),
    mpatches.Patch(color=COLORS["node"],      label="Fuera de lugar"),
    mpatches.Patch(color="#E5E7EB",           label="Espacio vacío"),
]
fig.legend(handles=leyenda, loc="lower center", ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.05))

plt.suptitle("Visualización del Puzzle de 8 Piezas", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## Sección 2: Heurísticas

Una **heurística** $h(n)$ estima el costo desde el estado $n$ hasta la meta. Para ser útil en A*, debe ser:
- **Admisible**: $h(n) \leq h^*(n)$ para todo $n$ (nunca sobreestima)
- **Consistente**: $h(n) \leq c(n, n') + h(n')$ para todo vecino $n'$

### H1: Fichas fuera de lugar
$$h_1(n) = \text{número de fichas en posición incorrecta (sin contar el espacio)}$$

### H2: Distancia Manhattan
$$h_2(n) = \sum_{i=1}^{8} |\text{fila}_i^{actual} - \text{fila}_i^{meta}| + |\text{col}_i^{actual} - \text{col}_i^{meta}|$$

In [ ]:
# ── Implementación de heurísticas ─────────────────────────────────────────────

def h_misplaced(state, meta=META):
    """
    H1: Fichas fuera de lugar (no cuenta el espacio vacío).
    """
    if isinstance(state, PuzzleState):
        state = state.state
    return sum(1 for i, v in enumerate(state) if v != 0 and v != meta[i])


def h_manhattan(state, meta=META):
    """
    H2: Suma de distancias Manhattan de cada ficha a su posición meta.
    """
    if isinstance(state, PuzzleState):
        state = state.state

    # Precomputa posición meta de cada valor
    pos_meta = {v: divmod(i, 3) for i, v in enumerate(meta)}

    total = 0
    for i, v in enumerate(state):
        if v == 0:
            continue
        fi_actual, ci_actual = divmod(i, 3)
        fi_meta,   ci_meta   = pos_meta[v]
        total += abs(fi_actual - fi_meta) + abs(ci_actual - ci_meta)

    return total


# ── Ejemplo con un estado ────────────────────────────────────────────────────
estado_ejemplo = PuzzleState((8, 1, 3, 4, 0, 2, 7, 6, 5))

print("Estado de ejemplo:")
for i in range(3):
    fila = estado_ejemplo.state[i*3:(i+1)*3]
    print("  ", " ".join(str(x) if x != 0 else "_" for x in fila))

print(f"\nh1 (fichas fuera de lugar): {h_misplaced(estado_ejemplo)}")
print(f"h2 (distancia Manhattan):   {h_manhattan(estado_ejemplo)}")
print(f"\nh2 >= h1: {h_manhattan(estado_ejemplo) >= h_misplaced(estado_ejemplo)} → Manhattan domina a Fichas Fuera de Lugar")

In [ ]:
# ── Demostración: h2 domina a h1 ─────────────────────────────────────────────
# Generamos 100 estados aleatorios y verificamos que h2(n) >= h1(n) siempre

N_TEST = 100
h1_vals, h2_vals = [], []
h2_mayor_h1 = 0

for i in range(N_TEST):
    estado = generate_solvable_puzzle(profundidad=random.randint(5, 25), semilla=i)
    h1 = h_misplaced(estado)
    h2 = h_manhattan(estado)
    h1_vals.append(h1)
    h2_vals.append(h2)
    if h2 >= h1:
        h2_mayor_h1 += 1

print(f"De {N_TEST} estados: h2 >= h1 en {h2_mayor_h1} casos ({100*h2_mayor_h1/N_TEST:.0f}%)")
print(f"Promedio h1: {np.mean(h1_vals):.2f}")
print(f"Promedio h2: {np.mean(h2_vals):.2f}")
print(f"\nConclusión: h_manhattan DOMINA a h_misplaced (h2(n) ≥ h1(n) siempre)")
print("Ambas son admisibles porque nunca pueden superar el costo real.")

# Visualización
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(N_TEST)
ax.plot(x, h2_vals, label="h2 Manhattan",  color=COLORS["primary"],   alpha=0.8, linewidth=1)
ax.plot(x, h1_vals, label="h1 Fuera lugar", color=COLORS["danger"],   alpha=0.8, linewidth=1)
ax.fill_between(x, h1_vals, h2_vals, alpha=0.15, color=COLORS["accent"],
                label="Diferencia h2 - h1")
ax.set_xlabel("Estado aleatorio")
ax.set_ylabel("Valor de heurística")
ax.set_title("Comparación de Heurísticas en 100 estados aleatorios", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

---
## Sección 3: BFS y A*

Implementamos los dos algoritmos de búsqueda y los probamos en una instancia de dificultad moderada.

In [ ]:
# ── Implementación de BFS ─────────────────────────────────────────────────────

def bfs_puzzle(inicio, meta=META):
    """
    BFS para el puzzle de 8 piezas.

    Retorna:
      camino     : lista de PuzzleState desde inicio hasta meta ([] si no hay)
      expandidos : número de nodos expandidos
    """
    if isinstance(inicio, tuple):
        inicio = PuzzleState(inicio)
    meta_state = PuzzleState(meta)

    if inicio == meta_state:
        return [inicio], 0

    # cola: (estado, camino)
    cola = deque([(inicio, [inicio])])
    visitados = {inicio}
    expandidos = 0

    while cola:
        estado, camino = cola.popleft()
        expandidos += 1

        for vecino, _, _ in estado.neighbors():
            if vecino == meta_state:
                return camino + [vecino], expandidos
            if vecino not in visitados:
                visitados.add(vecino)
                cola.append((vecino, camino + [vecino]))

    return [], expandidos


# ── Implementación de A* para el puzzle ──────────────────────────────────────

def a_star_puzzle(inicio, meta=META, h=h_manhattan):
    """
    A* para el puzzle de 8 piezas.

    Retorna:
      camino     : lista de PuzzleState desde inicio hasta meta
      expandidos : número de nodos expandidos
      max_frontera: tamaño máximo de la frontera (memoria)
    """
    if isinstance(inicio, tuple):
        inicio = PuzzleState(inicio)
    meta_state = PuzzleState(meta)

    # heap: (f, g, contador, estado, camino)
    contador = 0
    heap = [(h(inicio, meta), 0, contador, inicio, [inicio])]
    g_costs = {inicio: 0}
    visitados = set()
    expandidos = 0
    max_frontera = 1

    while heap:
        max_frontera = max(max_frontera, len(heap))
        f, g, _, estado, camino = heapq.heappop(heap)

        if estado in visitados:
            continue
        visitados.add(estado)
        expandidos += 1

        if estado == meta_state:
            return camino, expandidos, max_frontera

        for vecino, costo, _ in estado.neighbors():
            nuevo_g = g + costo
            if vecino not in visitados and (vecino not in g_costs or nuevo_g < g_costs[vecino]):
                g_costs[vecino] = nuevo_g
                contador += 1
                f_nuevo = nuevo_g + h(vecino, meta)
                heapq.heappush(heap, (f_nuevo, nuevo_g, contador, vecino, camino + [vecino]))

    return [], expandidos, max_frontera


print("BFS y A* implementados.")

In [ ]:
# ── Prueba en una instancia de profundidad ~10 ────────────────────────────────

inicio_test = generate_solvable_puzzle(profundidad=10, semilla=7)
print("Estado inicial:")
for i in range(3):
    fila = inicio_test.state[i*3:(i+1)*3]
    print("  ", " ".join(str(x) if x != 0 else "_" for x in fila))

# BFS
camino_bfs, exp_bfs = bfs_puzzle(inicio_test)
print(f"\nBFS:")
print(f"  Profundidad solución: {len(camino_bfs) - 1}")
print(f"  Nodos expandidos:     {exp_bfs}")

# A* + Manhattan
camino_astar, exp_astar, mem_astar = a_star_puzzle(inicio_test, h=h_manhattan)
print(f"\nA* (Manhattan):")
print(f"  Profundidad solución: {len(camino_astar) - 1}")
print(f"  Nodos expandidos:     {exp_astar}")
print(f"  Máximo en frontera:   {mem_astar}")

# A* + Misplaced
camino_miss, exp_miss, mem_miss = a_star_puzzle(inicio_test, h=h_misplaced)
print(f"\nA* (Fuera de lugar):")
print(f"  Profundidad solución: {len(camino_miss) - 1}")
print(f"  Nodos expandidos:     {exp_miss}")

In [ ]:
# ── Visualización del camino solución (A* + Manhattan) ────────────────────────

# Mostramos los primeros y últimos pasos del camino
pasos_mostrar = min(len(camino_astar), 8)  # máximo 8 pasos
indices = list(range(pasos_mostrar))
if len(camino_astar) > pasos_mostrar:
    # Si hay más de 8 pasos, mostramos inicio y fin
    mitad = pasos_mostrar // 2
    indices = list(range(mitad)) + list(range(len(camino_astar) - (pasos_mostrar - mitad), len(camino_astar)))

fig, axes = plt.subplots(1, len(indices), figsize=(2.5 * len(indices), 3.5))
if len(indices) == 1:
    axes = [axes]

for ax, idx in zip(axes, indices):
    titulo = f"Paso {idx}"
    if idx == 0:
        titulo = "Inicio"
    elif idx == len(camino_astar) - 1:
        titulo = "Meta"
    elif len(camino_astar) > pasos_mostrar and idx == indices[pasos_mostrar // 2]:
        titulo = f"...paso {idx}"
    draw_puzzle(camino_astar[idx], titulo=titulo, ax=ax)

plt.suptitle(f"Solución A* (Manhattan) — {len(camino_astar)-1} pasos",
              fontweight="bold", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

---
## Sección 4: Comparación de Heurísticas

Comparamos los tres algoritmos en instancias de dificultad creciente. Observaremos cómo el número de nodos expandidos crece exponencialmente con la profundidad, y cómo una mejor heurística lo reduce drásticamente.

In [ ]:
# ── Experimento: 5 instancias de dificultad creciente ────────────────────────

PROFUNDIDADES = [5, 10, 15, 20, 25]
SEMILLAS      = [1,  2,   3,  4,  5]

resultados = []

for prof, sem in zip(PROFUNDIDADES, SEMILLAS):
    estado_i = generate_solvable_puzzle(profundidad=prof, semilla=sem)

    # BFS (solo para instancias manejables)
    if prof <= 15:
        _, exp_b = bfs_puzzle(estado_i)
    else:
        exp_b = None  # demasiado lento

    _, exp_m, _ = a_star_puzzle(estado_i, h=h_misplaced)
    camino_man, exp_man, _ = a_star_puzzle(estado_i, h=h_manhattan)

    resultados.append({
        "prof":     prof,
        "prof_real": len(camino_man) - 1,
        "bfs":      exp_b,
        "miss":     exp_m,
        "manh":     exp_man,
    })

# Tabla de resultados
print(f"{'Prof.aprox':>10} {'Prof.real':>10} {'BFS':>12} {'A*(miss)':>12} {'A*(manh)':>12}")
print("-" * 60)
for r in resultados:
    bfs_str = str(r['bfs']) if r['bfs'] is not None else "(muy lento)"
    print(f"{r['prof']:>10} {r['prof_real']:>10} {bfs_str:>12} {r['miss']:>12} {r['manh']:>12}")

In [ ]:
# ── Gráfica de comparación ────────────────────────────────────────────────────

profs_reales = [r["prof_real"] for r in resultados]
exp_miss_list = [r["miss"]     for r in resultados]
exp_manh_list = [r["manh"]     for r in resultados]
exp_bfs_list  = [r["bfs"]      for r in resultados if r["bfs"] is not None]
profs_bfs     = [r["prof_real"] for r in resultados if r["bfs"] is not None]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Izquierda: escala lineal
axes[0].plot(profs_reales, exp_miss_list, "o-",
             color=COLORS["accent"],    label="A* Fuera de lugar", linewidth=2)
axes[0].plot(profs_reales, exp_manh_list, "s-",
             color=COLORS["secondary"], label="A* Manhattan",      linewidth=2)
if exp_bfs_list:
    axes[0].plot(profs_bfs, exp_bfs_list, "^-",
                 color=COLORS["danger"],  label="BFS",               linewidth=2)
axes[0].set_xlabel("Profundidad real de la solución")
axes[0].set_ylabel("Nodos expandidos")
axes[0].set_title("Nodos Expandidos (escala lineal)", fontweight="bold")
axes[0].legend()

# Derecha: escala logarítmica
axes[1].semilogy(profs_reales, exp_miss_list, "o-",
                  color=COLORS["accent"],    label="A* Fuera de lugar", linewidth=2)
axes[1].semilogy(profs_reales, exp_manh_list, "s-",
                  color=COLORS["secondary"], label="A* Manhattan",      linewidth=2)
if exp_bfs_list:
    axes[1].semilogy(profs_bfs, exp_bfs_list, "^-",
                      color=COLORS["danger"],  label="BFS",               linewidth=2)
axes[1].set_xlabel("Profundidad real de la solución")
axes[1].set_ylabel("Nodos expandidos (log)")
axes[1].set_title("Nodos Expandidos (escala log)", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()

# Bar chart por instancia
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(resultados))
ancho = 0.25
labels_x = [f"d={r['prof_real']}" for r in resultados]

b1 = ax.bar(x - ancho, exp_miss_list, ancho, label="A* Fuera de lugar",
             color=COLORS["accent"],    alpha=0.85)
b2 = ax.bar(x,          exp_manh_list, ancho, label="A* Manhattan",
             color=COLORS["secondary"], alpha=0.85)

ax.bar_label(b1, padding=3, fontsize=8)
ax.bar_label(b2, padding=3, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(labels_x)
ax.set_ylabel("Nodos expandidos")
ax.set_title("Comparación A* por Heurística en 5 Instancias", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

---
## Ejercicio 1

**Genera 10 instancias aleatorias solucionables del puzzle de 8. Para cada una, calcula el factor de ramificación efectivo $b^*$ para A\* con Manhattan. ¿Cuál es el $b^*$ promedio?**

El **factor de ramificación efectivo** $b^*$ se define como la raíz de la ecuación:

$$N + 1 = 1 + b^* + (b^*)^2 + \ldots + (b^*)^d$$

donde $N$ es el número de nodos generados (o expandidos) y $d$ es la profundidad de la solución. Una aproximación simple es:

$$b^* \approx N^{1/d}$$

Para el puzzle de 8 piezas con heurística Manhattan, se espera $b^* \approx 1.3$–$1.5$.

In [ ]:
# ── Ejercicio 1 ───────────────────────────────────────────────────────────────
# Pista: b* ≈ N^(1/d) donde N = nodos expandidos y d = profundidad solución
# Tu código aquí

# Paso 1: Genera 10 instancias con profundidades variadas (entre 10 y 25)
# instancias = [generate_solvable_puzzle(profundidad=random.randint(10, 25), semilla=i)
#               for i in range(10)]

# Paso 2: Para cada instancia, ejecuta A* con Manhattan
# b_estrella_vals = []
# for i, estado in enumerate(instancias):
#     camino, exp, _ = a_star_puzzle(estado, h=h_manhattan)
#     d = len(camino) - 1
#     N = exp
#     b_star = N ** (1/d) if d > 0 else 1.0
#     b_estrella_vals.append(b_star)
#     print(f"Instancia {i+1}: d={d}, N={N}, b*={b_star:.3f}")

# Paso 3: Calcula y muestra el b* promedio
# print(f"\nb* promedio: {np.mean(b_estrella_vals):.3f}")

---
## Ejercicio 2

**Implementa IDA\* para el puzzle de 8. Compara su uso de memoria (número máximo de estados en la pila) vs A\*. Prueba en una instancia de profundidad 20.**

IDA\* (Iterative Deepening A\*) funciona como una búsqueda en profundidad iterativa pero con un **límite de costo** que aumenta progresivamente:

1. Comienza con límite $f_0 = h(\text{inicio})$.
2. Realiza DFS, podando ramas con $f(n) > \text{límite}$.
3. Si no encuentra la meta, el nuevo límite es el mínimo valor $f$ que superó el límite anterior.
4. Repite hasta encontrar la meta.

**Ventaja de IDA\***: usa $O(d)$ memoria en lugar de $O(b^d)$ de A\*.

In [ ]:
# ── Ejercicio 2: IDA* ─────────────────────────────────────────────────────────
# Tu código aquí

# Estructura sugerida:
#
# def ida_star_puzzle(inicio, meta=META, h=h_manhattan):
#     """
#     IDA* para el puzzle de 8 piezas.
#     Retorna: (camino, nodos_expandidos, max_pila)
#     """
#
#     def busqueda(camino, g, limite, visitados_pila, max_pila_ref, expandidos_ref):
#         estado = camino[-1]
#         f = g + h(estado, meta)
#
#         if f > limite:
#             return f  # retorna el nuevo límite mínimo
#
#         if estado == PuzzleState(meta):
#             return -1  # señal de éxito
#
#         expandidos_ref[0] += 1
#         max_pila_ref[0] = max(max_pila_ref[0], len(camino))
#         minimo = float('inf')
#
#         for vecino, _, _ in estado.neighbors():
#             if vecino not in visitados_pila:
#                 visitados_pila.add(vecino)
#                 camino.append(vecino)
#                 resultado = busqueda(camino, g + 1, limite, visitados_pila, max_pila_ref, expandidos_ref)
#                 if resultado == -1:
#                     return -1
#                 if resultado < minimo:
#                     minimo = resultado
#                 camino.pop()
#                 visitados_pila.discard(vecino)
#
#         return minimo
#
#     # Inicialización
#     ...

# Prueba y comparación:
# estado_d20 = generate_solvable_puzzle(profundidad=20, semilla=99)
# camino_astar_d20, exp_a, mem_a = a_star_puzzle(estado_d20, h=h_manhattan)
# camino_ida, exp_i, mem_i = ida_star_puzzle(estado_d20)
#
# print(f"A*:   expandidos={exp_a}, max_frontera={mem_a}")
# print(f"IDA*: expandidos={exp_i}, max_pila={mem_i}")

---
## Ejercicio 3 (Desafío)

**El puzzle de 15 piezas tiene $16!/2 \approx 10^{13}$ estados. Estima cuántos nodos expandiría A\* con Manhattan vs IDA\* con Manhattan en una instancia de profundidad 50. Muestra tus cálculos.**

Pistas para el análisis teórico:
- Para A\*, el número de nodos es aproximadamente $O(b^{*d})$ donde $b^* \approx 1.3$–$1.5$ (valor empírico).
- Para IDA\*, en cada iteración con límite $k$ se expanden aproximadamente $O(b^k)$ nodos, pero la heurística Manhattan reduce $b^*$ significativamente.
- Usa los valores de $b^*$ que calculaste en el Ejercicio 1 como referencia.
- Para el puzzle de 15: la heurística Manhattan lineal da $b^* \approx 1.45$ según la literatura.

**Escribe tu análisis aquí:**

Con $b^* \approx 1.45$ y profundidad $d = 50$:

- **A\* estimado**: $N_{A^*} \approx (b^*)^d = ...$
- **IDA\* estimado**: cada iteración de IDA\* con límite $k$ expande $\approx (b^*)^k$ nodos, y hay $d$ iteraciones en el peor caso...

Completa los cálculos:

In [ ]:
# ── Ejercicio 3: Estimación teórica ──────────────────────────────────────────
# Tu código aquí

# Parámetros
# b_star = 1.45  # factor de ramificación efectivo para puzzle-15 con Manhattan
# d = 50         # profundidad de la instancia

# Estimación A*
# nodos_astar = b_star ** d
# print(f"A* estimado: {nodos_astar:.2e} nodos")

# Estimación IDA* (suma de nodos en cada iteración)
# En IDA*, el límite crece de h(inicio) hasta d (en pasos de ~1 para el puzzle)
# Suponiendo que h(inicio) ≈ d * 0.6 (heurística Manhattan para puzzle-15)
# h_inicio = d * 0.6
# nodos_ida = sum(b_star**k for k in range(int(h_inicio), d + 1))
# print(f"IDA* estimado: {nodos_ida:.2e} nodos")

# Visualización comparativa
# profundidades = range(10, 55, 5)
# nodos_a_list = [b_star**d for d in profundidades]
# nodos_i_list = [sum(b_star**k for k in range(int(d*0.6), d+1)) for d in profundidades]
# ...

pass